# GETTSIM Workshop 2026

## Stage 3 — Micro Data

This notebook is supposed to serve as a template for your own reform analysis using
microdata.

Here, we use the same example reform as yesterday. You use your custom reform from
yesterday instead.

The same reform, now on SOEP. Same order as yesterday: the two policy environments, the
targets, the input template GETTSIM derives from them — and then the part that is new,
filling that template from real survey data and stating what the survey does not
measure.

It stops at GETTSIM's output. Turning that output into a number is analysis, and the
choices it needs are yours.

In [ ]:
import numpy as np
import pandas as pd
import dags.tree as dt

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import PiecewisePolynomialParam, TTSIMUnit, get_piecewise_parameters
from soep_preparation.config import METADATA, MODULES
from soep_preparation.final_dataset import create_final_dataset

POLICY_DATE = "2023-07-01"
SURVEY_YEAR = 2023

### 1. The two policy environments

Create the status quo and reform policy environment here. The second cell is the only
one you need to change: paste the reform you built yesterday, or leave the demo reform
in place.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment, policy_date_str=POLICY_DATE
)

In [ ]:
reform = copy_environment(status_quo)
reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},
                {"interval": "[1000, 1200)", "slope": 0.1},
                {"interval": "[1200, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)

### 2. What `soep-preparation` gives you

Two things: **cleaned modules**, one tidy frame per SOEP file, variables renamed, typed,
and harmonised across waves, and a **metadata catalogue** that records, for every
variable, which module holds it, its dtype, and which period it describes.

Note: A mapping of SOEP variables to GETTSIM qnames is also provided, but it is not
complete enough to be useful here yet.

In [ ]:
sorted(METADATA)[0:10]

In [ ]:
entry = METADATA["rent_minus_heating_costs_m_hh"]
print(f"module:    {entry['module']}")
print(f"dtype:     {entry['dtype']}")
print(f"describes: {entry['reference']}")
print(f"waves:     {min(entry['survey_years'])}-{max(entry['survey_years'])}")

### 3. The targets

What you want back from GETTSIM, per person. The leaves are the column names the
results carry.

In [ ]:
TARGETS = {
    "bürgergeld": {"betrag_m_bg": "bürgergeld_m_bg"},
    "einkommensteuer": {"betrag_m_sn": "einkommensteuer_m_sn"},
    "sozialversicherung": {"beiträge_versicherter_m": "sozialversicherung_m"},
    "kindergeld": {"betrag_m": "kindergeld_m"},
    "wohngeld": {"betrag_m_wthh": "wohngeld_m_wthh"},
}

### 4. The input template

`MainTarget.templates.input_data_dtypes.tree` returns every input those targets need,
for that policy date. Ask for it rather than guessing: it is the authoritative list, and
it shrinks as soon as you override a node.

In [ ]:
without_overrides = dt.flatten_to_qnames(
    main(
        main_target=MainTarget.templates.input_data_dtypes.tree,
        policy_date_str=POLICY_DATE,
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
    )
)
without_overrides

That list asks for a contribution history, an Elterngeld biography and an
Arbeitslosengeld claim history per person. SOEP carries none of them.

But most of those inputs exist only to let GETTSIM compute the five benefit amounts
which *are* part of the SOEP. Hand it a node whose value you already know and it drops
that node's entire upstream subtree from the template.

In [ ]:
OVERRIDES = {
    "p_id": pd.Series([0]),
    "sozialversicherung": {
        "rente": {
            "altersrente": {"betrag_m": pd.Series([0.0])},
            "erwerbsminderung": {"betrag_m": pd.Series([0.0])},
        },
        "arbeitslosen": {"betrag_m": pd.Series([0.0])},
    },
    "elterngeld": {"betrag_m": pd.Series([0.0])},
    "unterhaltsvorschuss": {"betrag_m": pd.Series([0.0])},
}

needed = dt.flatten_to_qnames(
    main(
        main_target=MainTarget.templates.input_data_dtypes.tree,
        policy_date_str=POLICY_DATE,
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
        input_data=InputData.tree(OVERRIDES),
    )
)

print(f"inputs without the overrides: {len(without_overrides)}")
print(f"inputs with them:             {len(needed)}")

Those 53 are what the rest of this notebook has to produce: some read straight off a
SOEP variable, some derived from several, and the remainder — the ones SOEP does not
observe at all — supplied as stated constants.

In [ ]:
needed

### 5. The sample

Now go the other way: name the SOEP variables that will supply those inputs. The
metadata catalogue says which module each one lives in, so the module list is derived
rather than typed out, and `create_final_dataset` merges them into one frame for the
survey years you ask for.

Finding the variables is the part that takes time. Browse the
[documentation](https://soep-preparation.readthedocs.io/en/latest/variables/) of
soep-preparation.

In [ ]:
VARIABLES = [
    # Demographics.
    "age",
    "birth_year",
    "disability_degree",
    "federal_state_of_residence",
    # Labour market and income. The `_y` amounts describe the previous calendar year,
    # `gross_labor_income_previous_month_m` the month before the interview.
    "actual_working_hours_w",
    "gross_labor_income_previous_month_m",
    "earnings_from_self_employment_y",
    "gesetzliche_rente_y",
    "arbeitslosengeld_y",
    "in_education",
    # Household composition.
    "ehepartner_p_id",
    "partner_p_id",
    "p_id_mother",
    # Housing.
    "rent_minus_heating_costs_m_hh",
    "heating_costs_m_hh",
    "living_space_hh",
    "rented_or_owned",
]

module_names = sorted({str(METADATA[name]["module"]) for name in VARIABLES})

soep = create_final_dataset(
    modules={name: MODULES[name].load() for name in module_names},
    variables=VARIABLES,
    survey_years=[SURVEY_YEAR],
)
print(f"{len(soep):,} rows from: {', '.join(module_names)}")

That frame still holds everyone in the gross sample of `SURVEY_YEAR`, including people
who were not interviewed. Their inputs would be silently filled with zeros — which makes
them look poor rather than absent. `age` is asked in the interview, so requiring it
keeps the people who took part.

In [ ]:
soep = soep[soep["age"].notna()].reset_index(drop=True)
print(f"{len(soep):,} people in {soep['hh_id'].nunique():,} households")

We need to remove all missings from the input data before handing it to GETTSIM.

SOEP interviews from age 16, so a child's missing earnings are an absence rather than a
gap. The spouse and parent pointers are missing for people who have none; they turn into
`-1` in the next section.

In [ ]:
# Not asked, or asked and answered "none". The zero is the answer.
ZERO_IF_MISSING = [
    "disability_degree",
    "actual_working_hours_w",
    "gesetzliche_rente_y",
    "arbeitslosengeld_y",
]
ZERO_IF_NOT_INTERVIEWED = [
    "gross_labor_income_previous_month_m",
    "earnings_from_self_employment_y",
]
# Missing means "no such person", not "unobserved".
NO_SUCH_PERSON = ["ehepartner_p_id", "partner_p_id", "p_id_mother"]
INTERVIEW_AGE = 16

soep[ZERO_IF_MISSING] = soep[ZERO_IF_MISSING].fillna(0)
soep["in_education"] = soep["in_education"].fillna(value=False)
soep.loc[soep["age"] < INTERVIEW_AGE, ZERO_IF_NOT_INTERVIEWED] = 0

# Everything else has no defensible reading, so the whole household goes.
must_be_observed = [name for name in VARIABLES if name not in NO_SUCH_PERSON]
incomplete = set(soep.loc[soep[must_be_observed].isna().any(axis="columns"), "hh_id"])

soep = soep[~soep["hh_id"].isin(incomplete)].reset_index(drop=True)

print(f"Left with {len(soep):,} people in {soep['hh_id'].nunique():,} households")

### 6. More Data Cleaning and Derived Columns

Some variables need additonal cleaning to be able to be used with GETTSIM. Here, we
recode missing pointers to `-1` (GETTSIM placeholder for "no such person") and create
some additional derived columns.

Note: Eventually, this will be done in `soep-preparation`. We're just not quite there
yet.

In [ ]:
present = set(soep["p_id"])


def resolve_pointer(pointer):
    """Pointers to people outside the sample, and missing ones, become -1."""
    return pointer.where(pointer.isin(present), -1).fillna(-1)


soep["p_id_ehepartner"] = resolve_pointer(soep["ehepartner_p_id"])
soep["p_id_einstandspartner"] = resolve_pointer(soep["partner_p_id"])
soep["p_id_elternteil_1"] = resolve_pointer(soep["p_id_mother"])

spouse_of = dict(zip(soep["p_id"], soep["p_id_ehepartner"], strict=True))
soep["p_id_elternteil_2"] = soep["p_id_elternteil_1"].map(
    lambda p_id: spouse_of.get(p_id, -1)
)
# Kindergeld goes to the first parent on record.
soep["p_id_kindergeldempfänger"] = soep["p_id_elternteil_1"]

In [ ]:
EAST = {
    "Berlin",
    "Brandenburg",
    "Mecklenburg-Vorpommern",
    "Saxony",
    "Saxony-Anhalt",
    "Thuringia",
}
has_child = soep["p_id"].isin(
    pd.concat([soep["p_id_elternteil_1"], soep["p_id_elternteil_2"]])
)

# The interview date is unknown, so a birthday sits half a year into the year.
soep["alter_monate"] = soep["age"] * 12 + 6
soep["gemeinsam_veranlagt"] = soep["p_id_ehepartner"] >= 0
soep["alleinerziehend"] = has_child & (soep["p_id_ehepartner"] < 0)
soep["in_ausbildung"] = soep["in_education"] & (soep["age"] >= 18)
soep["bewohnt_eigentum_hh"] = soep["rented_or_owned"] == "Owner"
soep["wohnort_ost_hh"] = soep["federal_state_of_residence"].isin(EAST)

### 7. The mapper: data and defaults

Here, we finally bring data and assumptions together. The mapper maps SOEP variable
names to GETTSIM inputs and states the values of the inputs SOEP does not observe.

In [ ]:
MAPPER = {
    # --- identifiers and demographics ---------------------------------------
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "age",
    "alter_monate": "alter_monate",
    "geburtsjahr": "birth_year",
    "arbeitsstunden_w": "actual_working_hours_w",
    "behinderungsgrad": "disability_degree",
    # Merkzeichen G is not surveyed; only the Grad der Behinderung is.
    "schwerbehindert_grad_g": False,
    # SOEP surveys wealth only every five years, so it is unobserved in 2023. Zero
    # means everyone passes the Vermögensprüfung: this over-states entitlement. If
    # your reform touches the means test, this is the line to fix.
    "vermögen": 0.0,
    # --- family -------------------------------------------------------------
    "familie": {
        "alleinerziehend": "alleinerziehend",
        "p_id_ehepartner": "p_id_ehepartner",
        "p_id_elternteil_1": "p_id_elternteil_1",
        "p_id_elternteil_2": "p_id_elternteil_2",
    },
    # --- housing ------------------------------------------------------------
    "wohnen": {
        "bruttokaltmiete_m_hh": "rent_minus_heating_costs_m_hh",
        "heizkosten_m_hh": "heating_costs_m_hh",
        "wohnfläche_hh": "living_space_hh",
        "bewohnt_eigentum_hh": "bewohnt_eigentum_hh",
    },
    "wohnort_ost_hh": "wohnort_ost_hh",
    "wohngeld": {
        # SOEP has no Gemeindekennziffer, so the statutory Mietstufe cannot be
        # assigned. 3 is the modal Stufe. This one matters — see the Wohnkosten talk.
        "mietstufe_hh": 3,
    },
    # --- Bürgergeld ---------------------------------------------------------
    "bürgergeld": {
        "p_id_einstandspartner": "p_id_einstandspartner",
        # Decides which Vermögensfreibetrag applies. Does not bind here, because
        # `vermögen` is zero for everyone.
        "bezug_im_vorjahr": True,
    },
    "kindergeld": {
        "p_id_empfänger": "p_id_kindergeldempfänger",
        "in_ausbildung": "in_ausbildung",
    },
    # --- income -------------------------------------------------------------
    "einnahmen": {
        "bruttolohn_m": "gross_labor_income_previous_month_m",
        # Household-level and top-coded in SOEP; below the Sparerpauschbetrag for
        # most Bürgergeld households anyway.
        "kapitalerträge_y": 0.0,
        "renten": {
            "aus_berufsständischen_versicherungen_m": 0.0,  # rare
            "basisrente_m": 0.0,  # not separable in SOEP
            "betriebliche_altersvorsorge_m": 0.0,  # not separable in SOEP
            "geförderte_private_vorsorge_m": 0.0,  # not separable in SOEP
            "sonstige_private_vorsorge_m": 0.0,  # not separable in SOEP
        },
    },
    "einkommensteuer": {
        "gemeinsam_veranlagt": "gemeinsam_veranlagt",
        "abzüge": {
            # Riester/Rürup contributions not separable in SOEP.
            "beitrag_private_rentenversicherung_m": 0.0,
            # Childcare costs are surveyed irregularly; zero understates deductions.
            "kinderbetreuungskosten_m": 0.0,
            "p_id_kinderbetreuungskostenträger": -1,  # follows from the line above
        },
        "einkünfte": {
            "aus_selbstständiger_arbeit": {
                "betrag_y": "earnings_from_self_employment_y"
            },
            # Negligible for the working-age population Bürgergeld concerns.
            "aus_forst_und_landwirtschaft": {"betrag_y": 0.0},
            # SOEP self-employment income is not split by Einkunftsart; booked as
            # selbstständige Arbeit instead.
            "aus_gewerbebetrieb": {"betrag_y": 0.0},
            "aus_nichtselbstständiger_arbeit": {
                # Not surveyed; GETTSIM applies the Arbeitnehmerpauschbetrag.
                "tatsächliche_werbungskosten_y": 0.0,
            },
            # Rental income is surveyed at household level only.
            "aus_vermietung_und_verpachtung": {"betrag_y": 0.0},
            # Affects health-insurance treatment only; rare in this population.
            "ist_hauptberuflich_selbstständig": False,
            "sonstige": {
                "alle_weiteren_y": 0.0,  # residual category
                "rente": {
                    # Only binds when private pension income is non-zero, which it
                    # is not here.
                    "alter_beginn_leistungsbezug_sonstige_private_vorsorge": 67,
                },
            },
        },
    },
    # Maintenance received is not reliably measured in SOEP.
    "unterhalt": {"tatsächlich_erhaltener_betrag_m": 0.0},
    # --- the five overridden nodes ------------------------------------------
    # Take-up is not observed.
    "unterhaltsvorschuss": {"betrag_m": 0.0},
    # Eligibility needs a birth history and prior net income; out of scope for a
    # Bürgergeld reform.
    "elterngeld": {"betrag_m": 0.0},
    "sozialversicherung": {
        # Private insurance is rare among Bürgergeld-eligible households.
        "kranken": {"beitrag": {"privat_versichert": False}},
        # Avoids the childless surcharge; refine if your reform is sensitive to it.
        "pflege": {"beitrag": {"hat_kinder": True}},
        "arbeitslosen": {"betrag_y": "arbeitslosengeld_y"},
        "rente": {
            "bezieht_rente": False,  # set together with the overridden amount
            # Far future, so no one is treated as retired.
            "jahr_renteneintritt": 2080,
            "altersrente": {"betrag_y": "gesetzliche_rente_y"},
            # Disability pension receipt is not cleanly identified in SOEP.
            "erwerbsminderung": {"betrag_m": 0.0},
            # Grundrente needs a full contribution history.
            "grundrente": {"grundrentenzeiten_monate": 0},
        },
    },
}

The mapper should cover exactly the inputs section 4 said were needed.

In [ ]:
supplied = set(dt.flatten_to_qnames(MAPPER))
missing = sorted(set(needed) - supplied)
spurious = sorted(supplied - set(needed))

print(f"missing:  {missing}")
print(f"spurious: {spurious}")

### 8. Running the reform

`InputData.df_and_mapper` hands GETTSIM the frame and the mapper together;
`MainTarget.results.df_with_mapper` returns the targets under the names you gave them.

In [ ]:
def run(policy_environment):
    return main(
        main_target=MainTarget.results.df_with_mapper,
        policy_date_str=POLICY_DATE,
        policy_environment=policy_environment,
        input_data=InputData.df_and_mapper(df=soep, mapper=MAPPER),
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
    )


before = run(status_quo)
after = run(reform)
before.head()